In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import pickle

In [2]:
#loading the dataset
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#preprocessing the data
#dropping unnecessary columns
data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
#label encoding the categorical variables
label_encoder=LabelEncoder()
data["Gender"]=label_encoder.fit_transform(data["Gender"])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
##one-hot encoding the "Geography" column
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo=OneHotEncoder()
geo_encoder=one_hot_encoder_geo.fit_transform(data[["Geography"]])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [6]:
one_hot_encoder_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
geo_eoncoded_df=pd.DataFrame(geo_encoder.toarray(),columns=one_hot_encoder_geo.get_feature_names_out())
geo_eoncoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [9]:
#combining the one-hot encoded dataframe with the original dataframe
data=pd.concat([data.drop("Geography",axis=1),geo_eoncoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [10]:
#save the encoder and scaler for future use
with open("label_encoder.pkl","wb")as file:
    pickle.dump(label_encoder,file)

with open("one_hot_encoder_geo.pkl","wb")as file:
    pickle.dump(one_hot_encoder_geo,file)


In [11]:
#divide data into features and target variable
x=data.drop(["Exited"],axis=1)
y=data["Exited"]
#split the data into training and testing sets
X_train,X_test,Y_train,Y_test=train_test_split(x,y,test_size=0.2,random_state=42)
#feature scaling
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [12]:
#save the scaler for future use
with open("scaler.pkl","wb")as file:
    pickle.dump(scaler,file)

ANN IMPLEMENTION 

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [14]:
#building the ANN model
model=Sequential([
    Dense(64,activation="relu",input_shape=(X_train.shape[1],)),#Hidden layer 1 connected to the input layer
    Dense(32,activation="relu"),#Hidden layer 2
    Dense(1,activation="sigmoid")#Output layer
])

In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.001)
loss=tensorflow.keras.losses.BinaryCrossentropy()

In [17]:
#compile the model
model.compile(optimizer=opt,loss=loss,metrics=["Accuracy"])

In [18]:
#set up the tensorboard callback
from tensorflow.keras.callbacks import TensorBoard,EarlyStopping
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [19]:
#set up early stopping
early_stopping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [20]:
#train the model
history=model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,callbacks=[ tensorflow_callback,early_stopping_callback])

Epoch 1/100


250/250 [==============================] - 2s 5ms/step - loss: 0.4474 - Accuracy: 0.8106 - val_loss: 0.3937 - val_Accuracy: 0.8280
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3865 - Accuracy: 0.8384 - val_loss: 0.3575 - val_Accuracy: 0.8555
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3556 - Accuracy: 0.8530 - val_loss: 0.3464 - val_Accuracy: 0.8590
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3442 - Accuracy: 0.8574 - val_loss: 0.3451 - val_Accuracy: 0.8545
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3395 - Accuracy: 0.8577 - val_loss: 0.3429 - val_Accuracy: 0.8560
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3357 - Accuracy: 0.8621 - val_loss: 0.3437 - val_Accuracy: 0.8550
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3336 - Accuracy: 0.8626 - val_loss: 0.3432 - val_Accuracy: 0.85

In [21]:
model.save('model.h5')

c:\Users\hband\anaconda3\envs\tfenv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [22]:
#load tensorboard Extension
%load_ext tensorboard


In [24]:
%tensorboard --logdir logs/fit